In [1]:
!ls ../data/amazon_polarity/*/amazon*.jsonl

../data/amazon_polarity/Gemma4E4B/amazon_annotated1.jsonl
../data/amazon_polarity/Qwen36-27B/amazon_annotated1.jsonl
../data/amazon_polarity/Qwen36-27B/amazon_annotated2.jsonl
../data/amazon_polarity/Qwen36-27B/amazon_annotated3.jsonl


In [2]:
import json
import pandas as pd
from glob import glob
from pathlib import Path

PATH = "../data/amazon_polarity/*/amazon*.jsonl"
files = sorted(glob(PATH))

def load_json(file):
    model = Path(file).parent.name
    def get_data(raw):
        line = json.loads(raw)
        return {
            "text": line["text"],
            "labels": line.get("labels"),
            "not_labels": line.get("not_labels"),
            "model": model,
        }
    with open(file, "r") as f:
        data = [get_data(line) for line in f]
    return data

files_data = [load_json(file) for file in files]
df = pd.DataFrame([i for file_data in files_data for i in file_data])

In [3]:
df.sample(5)

,text,labels,not_labels,model
20253,Beautiful volume\n\nI was more than pleased wi...,"[unqualified_satisfaction_expression, expresse...","[detailed_plot_analysis_provided, critique_of_...",Gemma4E4B
17998,Just a waste of time\n\nLightweight horror yar...,"[dismissive_overall_assessment, critical_perfo...","[recommendation_for_viewing, detailed_plot_sum...",Gemma4E4B
34401,an immense story\n\nTHE DAY THE WORLD CAME TO ...,"[elevated_narrative_framing, macro_historical_...","[detailed_plot_summary_provision, technical_ed...",Gemma4E4B
35552,confessions\n\nConfessions was a thought provo...,"[thought_provoking_assessment, intellectual_en...","[detailed_plot_summary_offering, critique_of_p...",Gemma4E4B
22171,Worst Book Ever\n\nI had to read this book for...,"[forced_reading_complaint, strong_disapproval_...","[detailed_plot_summary_provided, comparison_to...",Gemma4E4B


In [4]:
def merge_group(group):
    merged_labels = set().union(*group["labels"])
    merged_not_labels = set().union(*group["not_labels"])
    merged_not_labels -= merged_labels  # remove any intersection
    models = list(sorted(set(group["model"])))[0]
    return pd.Series({
        "labels": sorted(merged_labels),
        "not_labels": sorted(merged_not_labels),
        "model": models if len(models) > 1 else models[0],
    })

df = (
    df.groupby("text", sort=False)
    .apply(merge_group, include_groups=False)
    .reset_index()
)
print(f"{len(df)} unique texts after merging")
df.sample(5)


50628 unique texts after merging


,text,labels,not_labels,model
21676,Surprises with every turn of the page!\n\nI wa...,"[anticipatory_future_reading, authorial_skill_...","[ambiguous_opinion_expression, binding_quality...",Gemma4E4B
7214,Best Haunted House CGI I've Seen So Far\n\nI l...,"[acknowledgement_of_original_merit, acknowledg...","[actor_chemistry_assessment, audience_reaction...",Gemma4E4B
33209,Wrong Version of The Year Without a Santa Clau...,"[absolute_rejection_posture, accusatory_vendor...","[acknowledgement_of_value, ambiguous_sentiment...",Gemma4E4B
27707,Best Novel Of 20th Century?\n\nIs there any do...,"[absolute_quality_assertion, aspirational_lite...","[academic_deconstruction_attempt, accessibilit...",Gemma4E4B
25884,Amazing film\n\nI just have to say that everyt...,"[affective_resonance_seeking, argument_of_subs...","[acting_performance_detail, actor_chemistry_as...",Gemma4E4B


In [5]:
import datasets

train_ds = datasets.Dataset.from_pandas(df)

dataset = datasets.DatasetDict({
    "train": train_ds
})

dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'not_labels', 'model'],
        num_rows: 50628
    })
})

In [6]:
dataset.push_to_hub("alexneakameni/ZSHOT-HARDSET-Polarity", commit_description="Upload of ZSHOT-HARDSET-Polarity with train/test split based on held-out labels.")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/alexneakameni/ZSHOT-HARDSET-Polarity/commit/4ab40c183302d948526f836a8aa08abc7af634ac', commit_message='Upload dataset', commit_description='Upload of ZSHOT-HARDSET-Polarity with train/test split based on held-out labels.', oid='4ab40c183302d948526f836a8aa08abc7af634ac', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/alexneakameni/ZSHOT-HARDSET-Polarity', endpoint='https://huggingface.co', repo_type='dataset', repo_id='alexneakameni/ZSHOT-HARDSET-Polarity'), pr_revision=None, pr_num=None)